# Ultrasonic Doppler Gesture Recognition — SVM vs Random Forest

**Ready-to-run Google Colab notebook** · 7 code cells · no paths to type.

This uses your existing `gesture_dataset_50_samples.zip`, including its Windows-style `\` ZIP names. It preserves all original WAV/CSV files. We train a **Support Vector Machine (SVM) with an RBF kernel**, which is often well suited to small datasets with engineered numerical features. To make comparison fair, an optional later cell evaluates your **Random Forest** on the **same features and identical held-out recording sessions**.

**How to run:** In Colab, use **File → Upload notebook**. Run **Cells 1–7 in order**. First run **Cell 1**, which provides an upload button when the ZIP isn't already present. **Never use the final all-data model to report test accuracy**—the evaluation uses the previous held-out-session folds.

**Data:** 2 sessions × 25 recordings = 50 WAV files; 5 gesture classes; 48 kHz, 4 channels (features use stronger microphones 1 and 2).

## Cell 1 — Upload your original ZIP, with Windows ZIP path fix

Run ▶. Select `gesture_dataset_50_samples.zip` if Colab asks. If that file is already present in `/content/`, the notebook reuses it without asking. This cell fixes the Windows `\` separators in the ZIP. Expect **50 WAV files + 2 labels.csv files**. This only modifies a temporary extracted copy.

In [ ]:
# CELL 1 — UPLOAD YOUR DATASET ZIP (WINDOWS ZIP FIX)
# You can also reuse a ZIP previously uploaded during this Colab session.
from google.colab import files
from pathlib import Path, PurePosixPath
from collections import Counter
import shutil, zipfile, re

DATA_DIR = Path('/content/ultrasonic_dataset')
expected_name = Path('/content/gesture_dataset_50_samples.zip')

# Reuse your already uploaded dataset if it is still available in Colab.
if expected_name.is_file():
    zip_paths = [expected_name]
    print('Using the already uploaded ZIP:', expected_name.name)
else:
    print('Click Choose Files and select your dataset ZIP.')
    uploaded = files.upload()
    zip_paths = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_paths:
        raise ValueError('Please upload a .zip dataset file.')

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)  # Delete only the extracted temporary copy.
DATA_DIR.mkdir(parents=True)

def safe_extract_windows_zip(zip_path, output_dir):
    """Extract while interpreting Windows backslashes as folder separators."""
    output_dir.mkdir(parents=True, exist_ok=True)
    root = output_dir.resolve()
    with zipfile.ZipFile(zip_path, 'r') as archive:
        for info in archive.infolist():
            normalized = info.filename.replace('\\', '/')
            name = PurePosixPath(normalized)
            # Do not permit absolute paths or traversal out of the output folder.
            if (name.is_absolute() or '..' in name.parts or
                    any(':' in part for part in name.parts)):
                raise ValueError(f'Unsafe entry in ZIP: {info.filename}')
            target = root.joinpath(*name.parts)
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe path: {info.filename}')
            if info.is_dir() or normalized.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, open(target, 'wb') as dest:
                    shutil.copyfileobj(source, dest)

for zp in zip_paths:
    folder = DATA_DIR if len(zip_paths) == 1 else DATA_DIR / zp.stem
    safe_extract_windows_zip(zp, folder)

# Handle nested archives too, if the dataset ZIP includes another ZIP.
for _ in range(2):
    nested = sorted(p for p in DATA_DIR.rglob('*.zip') if p.is_file())
    if not nested:
        break
    for nested_zip in nested:
        safe_extract_windows_zip(nested_zip, nested_zip.with_suffix(''))
        nested_zip.unlink()  # Temporary extracted copy only.

# Standardize duplicate CSV naming only if necessary, preserving the source CSV.
for p in DATA_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() == '.csv':
        if re.fullmatch(r'labels(?:\s*\(\d+\))?', p.stem, re.I):
            target = p.with_name('labels.csv')
            if not target.exists():
                shutil.copy2(p, target)

wav_paths = sorted(p for p in DATA_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')
csv_paths = sorted(p for p in DATA_DIR.rglob('*') if p.is_file() and p.name.lower() == 'labels.csv')
print('\n===== DATASET CHECK =====')
print('WAV files found:', len(wav_paths))
print('Labels CSV files found:', len(csv_paths))
for csv in csv_paths:
    print('Labels:', csv.relative_to(DATA_DIR))
if len(wav_paths) != 50 or len(csv_paths) != 2:
    raise ValueError('Expected 50 WAV files and 2 labels.csv files. Check the listing above.')
print('\nSUCCESS! Run Cell 2 next to read the WAV files and labels.')


## Cell 2 — Read the CSV labels and validate the WAV files

Run ▶. The model will learn from the **ground-truth gesture labels in each `labels.csv`** (not from filename assumptions or hand identity). Check that you see five examples of each gesture in each session.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import wavfile

CLASSES = ['LEFT', 'RIGHT', 'TOWARD', 'AWAY', 'IDLE']
LABEL_NAMES = ['gesture', 'label', 'class', 'gesture_label', 'gesture_type',
               'class_label', 'direction', 'movement', 'action', 'gesture_name']
FILENAME_NAMES = ['filename', 'file_name', 'file', 'wav_file', 'wav_filename',
                  'audio_file', 'audio_filename', 'recording_file',
                  'recording_filename', 'file_path', 'wav_path',
                  'audio_path', 'recording_path', 'path', 'recording', 'wav']

def clean_label(value):
    label = str(value).strip().upper()
    aliases = {'TOWARDS': 'TOWARD', 'FORWARD': 'TOWARD',
               'BACKWARD': 'AWAY', 'BACK': 'AWAY', 'STILL': 'IDLE', 'NONE': 'IDLE'}
    return aliases.get(label, label)

def get_label_column(frame):
    mapping = {str(c).strip().lower(): c for c in frame.columns}
    for name in LABEL_NAMES:
        if name in mapping:
            candidate = mapping[name]
            valid = frame[candidate].map(clean_label).isin(CLASSES).mean()
            if valid >= 0.8:
                return candidate
    for c in frame.columns:
        if str(c).strip().lower() == 'hand':
            continue
        valid = frame[c].map(clean_label).isin(CLASSES).mean()
        if valid >= 0.8:
            return c
    raise ValueError(f'Could not identify gesture label column among {list(frame.columns)}')

def get_filename_column(frame):
    mapping = {str(c).strip().lower(): c for c in frame.columns}
    ordered = [mapping[n] for n in FILENAME_NAMES if n in mapping]
    ordered += [c for c in frame.columns if c not in ordered]
    for c in ordered:
        values = frame[c].dropna().astype(str)
        if len(values) and values.str.lower().str.contains(r'\.wav(?:$|[\\/])', regex=True).any():
            return c
    # Permit explicit filename/path columns with extension-free names.
    for n in FILENAME_NAMES:
        if n in mapping:
            return mapping[n]
    raise ValueError('No WAV filename column identified. See the printed CSV headers/rows.')

def get_session_root(csv_file):
    # A session directory may contain labels.csv in a metadata subfolder.
    for parent in csv_file.parents:
        if parent == DATA_DIR.parent:
            break
        if parent.name in ('20261106_015202_P01', '20261106_021610_P01'):
            return parent
    return csv_file.parent

records = []
for csv_file in csv_paths:
    labels = pd.read_csv(csv_file, encoding='utf-8-sig')
    print(f'\n{csv_file.relative_to(DATA_DIR)} — columns: {list(labels.columns)}')
    display(labels.head(3))
    label_col = get_label_column(labels)
    file_col = get_filename_column(labels)
    session_root = get_session_root(csv_file)
    session_wavs = [p for p in wav_paths if p.is_relative_to(session_root)]
    if not session_wavs:
        raise ValueError(f'No WAV files found under session directory {session_root}')
    for i, row in labels.iterrows():
        label = clean_label(row[label_col])
        if label not in CLASSES:
            raise ValueError(f'Unknown gesture label {row[label_col]!r}, row {i+1} in {csv_file}')
        raw_filename = str(row[file_col]).strip().replace('\\', '/')
        name = Path(raw_filename).name.lower()
        exact = [p for p in session_wavs if p.name.lower() == name]
        matches = exact or [p for p in session_wavs if p.stem.lower() == Path(name).stem]
        if len(matches) != 1:
            raise ValueError(f'Cannot uniquely match {raw_filename!r} in {csv_file.name}: '
                             f'{len(matches)} candidates. Adjust the filename column/mapping.')
        records.append({'path': str(matches[0]), 'gesture': label,
                        'session': session_root.name, 'csv': str(csv_file)})

metadata = pd.DataFrame(records)
if metadata['path'].duplicated().any():
    raise ValueError('The same WAV was matched to multiple CSV rows. Check the label files.')
print('\nLabeled recordings:', len(metadata))
print('Gesture counts per session:')
display(pd.crosstab(metadata['session'], metadata['gesture']).reindex(columns=CLASSES, fill_value=0))
if metadata['session'].nunique() != 2:
    raise ValueError('Expected exactly two different recording sessions for session-based validation.')
if len(metadata) != 50:
    print('WARNING: expected 50 total recordings; review the CSVs and WAV files.')
if len(metadata) != len(wav_paths):
    print(f'WARNING: {len(wav_paths)-len(metadata)} WAV files were not matched to a label.')

print('\nSample audio checks:')
for session, part in metadata.groupby('session'):
    p = part.iloc[0]['path']
    sr, audio = wavfile.read(p)
    channels = audio.shape[1] if audio.ndim == 2 else 1
    print(f'{session}: {sr} Hz, {channels} channels, {len(audio)/sr:.2f} s, {audio.dtype}')
    if sr != 48000 or channels < 2:
        raise ValueError(f'Expected 48 kHz and at least 2 channels: {p}')

## Cell 3 — STFT ultrasonic Doppler feature extraction

Run ▶. Each complete WAV recording becomes **one numerical feature vector**. We measure the upper/lower Doppler sidebands around **18.5 and 19.0 kHz**, using microphones 1 and 2. This is **identical to your Random Forest notebook**, to make the comparison fair.

In [ ]:
from scipy.signal import stft

CARRIERS = (18500.0, 19000.0)
CHANNELS = (0, 1)                    # zero-based indices: microphones 1 and 2
SIDE_BAND_LOW_HZ = 12.0
SIDE_BAND_HIGH_HZ = 150.0
FFT_SIZE = 8192                     # ~5.86 Hz frequency resolution at 48 kHz
HOP_SIZE = 2048
EPS = 1e-15


def summary_features(values, prefix):
    values = np.asarray(values, dtype=np.float64)
    return {
        f'{prefix}_mean': float(np.mean(values)),
        f'{prefix}_std': float(np.std(values)),
        f'{prefix}_median': float(np.median(values)),
        f'{prefix}_p10': float(np.percentile(values, 10)),
        f'{prefix}_p90': float(np.percentile(values, 90)),
    }


def extract_features(wav_path):
    sr, audio = wavfile.read(wav_path)
    if sr != 48000:
        raise ValueError(f'Expected 48 kHz, found {sr} in {wav_path}')
    if audio.ndim == 1:
        raise ValueError(f'Expected 4-channel (at least 2-channel) audio: {wav_path}')
    if audio.shape[1] < 2:
        raise ValueError(f'Only {audio.shape[1]} channel(s) in {wav_path}')
    if len(audio) < FFT_SIZE:
        raise ValueError(f'Recording shorter than FFT window: {wav_path}')

    if np.issubdtype(audio.dtype, np.integer):
        scale = max(abs(np.iinfo(audio.dtype).min), np.iinfo(audio.dtype).max)
        signal = audio[:, list(CHANNELS)].astype(np.float64) / float(scale)
    else:
        signal = audio[:, list(CHANNELS)].astype(np.float64)

    features = {}
    for k, channel in enumerate(CHANNELS):
        x = signal[:, k]
        freqs, times, Z = stft(x, fs=sr, window='hann', nperseg=FFT_SIZE,
                               noverlap=FFT_SIZE-HOP_SIZE, boundary=None,
                               padded=False)
        power = np.abs(Z) ** 2
        for carrier in CARRIERS:
            offset = freqs - carrier
            pos_mask = (offset >= SIDE_BAND_LOW_HZ) & (offset <= SIDE_BAND_HIGH_HZ)
            neg_mask = (offset <= -SIDE_BAND_LOW_HZ) & (offset >= -SIDE_BAND_HIGH_HZ)
            center_mask = np.abs(offset) <= SIDE_BAND_LOW_HZ
            if not (pos_mask.any() and neg_mask.any() and center_mask.any()):
                raise RuntimeError('Frequency masks empty; verify sample rate / FFT configuration.')
            pos = power[pos_mask].sum(axis=0)
            neg = power[neg_mask].sum(axis=0)
            center = power[center_mask].sum(axis=0)

            # Ratios reduce sensitivity to amplitude and microphone gain.
            balance = np.log10((pos + EPS) / (neg + EPS))
            activity = np.log10((pos + neg + EPS) / (center + EPS))
            positive_activity = np.log10((pos + EPS) / (center + EPS))
            negative_activity = np.log10((neg + EPS) / (center + EPS))
            prefix = f'ch{channel+1}_f{int(carrier)}'
            features.update(summary_features(balance, f'{prefix}_balance'))
            features.update(summary_features(activity, f'{prefix}_activity'))
            features[f'{prefix}_pos_p90'] = float(np.percentile(positive_activity, 90))
            features[f'{prefix}_neg_p90'] = float(np.percentile(negative_activity, 90))
            features[f'{prefix}_pos_median'] = float(np.median(positive_activity))
            features[f'{prefix}_neg_median'] = float(np.median(negative_activity))
            features[f'{prefix}_fraction_pos'] = float(np.mean(balance > 0))
            mid = max(1, len(balance) // 2)
            features[f'{prefix}_balance_first_half'] = float(np.mean(balance[:mid]))
            features[f'{prefix}_balance_second_half'] = float(np.mean(balance[mid:]))

    return features

feature_rows = []
for i, record in metadata.iterrows():
    feature_rows.append(extract_features(record['path']))
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(metadata)} recordings')

X_df = pd.DataFrame(feature_rows)
if not np.isfinite(X_df.to_numpy()).all():
    raise ValueError('Nonfinite features detected; inspect audio for corruption.')
y = metadata['gesture'].to_numpy()
groups = metadata['session'].to_numpy()
print(f'Features extracted: X shape = {X_df.shape}, y shape = {y.shape}')
display(X_df.iloc[:3, :10])

## Cell 4 — Train the **SVM (RBF)**, test on the other session

Run ▶. First, train on Session 1 and test on Session 2, then swap them. We scale the features **inside** each training fold with `StandardScaler`—this is especially important for SVM and avoids leakage from the test session. Fixed starter settings: RBF kernel, `C=1.0`, `gamma='scale'`, `class_weight='balanced'`. Do not tune these against the final held-out scores.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.base import clone

# Start with fixed settings (no hyperparameter search on held-out test sessions).
svm_pipeline = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
)

X = X_df.to_numpy(dtype=np.float64)
cv = LeaveOneGroupOut()  # One full recording session is unseen in each fold.
predictions_svm = np.empty(len(y), dtype=object)
svm_folds = []

for train_idx, test_idx in cv.split(X, y, groups):
    model = clone(svm_pipeline)
    model.fit(X[train_idx], y[train_idx])
    predictions_svm[test_idx] = model.predict(X[test_idx])
    session = groups[test_idx[0]]
    svm_folds.append({
        'Unseen test session': session,
        'Train WAVs': len(train_idx),
        'Test WAVs': len(test_idx),
        'Accuracy': accuracy_score(y[test_idx], predictions_svm[test_idx]),
        'Macro precision': precision_score(y[test_idx], predictions_svm[test_idx],
                                           average='macro', labels=CLASSES, zero_division=0),
        'Macro recall': recall_score(y[test_idx], predictions_svm[test_idx],
                                     average='macro', labels=CLASSES, zero_division=0),
        'Macro F1': f1_score(y[test_idx], predictions_svm[test_idx],
                             average='macro', labels=CLASSES, zero_division=0)
    })

svm_fold_results = pd.DataFrame(svm_folds)
print('SVM (RBF): Train on Session A → Test on B, then reverse')
display(svm_fold_results.style.format({
    'Accuracy': '{:.1%}', 'Macro precision': '{:.1%}',
    'Macro recall': '{:.1%}', 'Macro F1': '{:.1%}',
}))

## Cell 5 — SVM accuracy, precision, recall, F1 and confusion matrix

Run ▶. You'll see the **actual held-out SVM scores**, a five-class classification report and the confusion matrix. Rows = **true gestures**; columns = **predicted gestures**. For each incorrect prediction, see the corresponding WAV filename.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print('=== SVM (RBF): HELD-OUT SESSION RESULTS ===')
print(f'Accuracy:        {accuracy_score(y, predictions_svm):.2%}')
print(f'Macro precision: {precision_score(y, predictions_svm, average="macro", labels=CLASSES, zero_division=0):.2%}')
print(f'Macro recall:    {recall_score(y, predictions_svm, average="macro", labels=CLASSES, zero_division=0):.2%}')
print(f'Macro F1-score:  {f1_score(y, predictions_svm, average="macro", labels=CLASSES, zero_division=0):.2%}')
print('\nPer-gesture results:')
print(classification_report(y, predictions_svm, labels=CLASSES, target_names=CLASSES,
                            digits=3, zero_division=0))

svm_cm = confusion_matrix(y, predictions_svm, labels=CLASSES)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=svm_cm, display_labels=CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d', xticks_rotation=30
)
ax.set_title('SVM (RBF): Leave-one-session-out confusion matrix')
plt.tight_layout()
fig.savefig('/content/svm_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

svm_report = pd.DataFrame(classification_report(
    y, predictions_svm, labels=CLASSES, target_names=CLASSES,
    output_dict=True, zero_division=0)).T
svm_report.to_csv('/content/svm_classification_report.csv')

svm_prediction_table = pd.DataFrame({
    'wav': [Path(p).name for p in metadata['path']],
    'session': groups, 'actual': y,
    'predicted': predictions_svm,
    'correct': y == predictions_svm
})
svm_prediction_table.to_csv('/content/svm_predictions.csv', index=False)
print('Most difficult gestures (lowest F1 first):')
display(svm_report.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']]
        .sort_values('f1-score'))
print('Misclassified recordings:')
display(svm_prediction_table.loc[~svm_prediction_table['correct']])

## Cell 6 — **Fair head-to-head comparison** with Random Forest

Run ▶. We train the original **Random Forest** again with exactly the settings used in your earlier notebook (300 trees, `max_depth=5`, `min_samples_leaf=2`, balanced classes), using the **same 50 recordings, the same extracted features, and the same train/test session folds**. The resulting table is the most defensible direct comparison between your two models. **Do not compare accuracy from different dataset splits.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
predictions_rf = np.empty(len(y), dtype=object)
rf_folds = []
for train_idx, test_idx in cv.split(X, y, groups):
    model = clone(random_forest)
    model.fit(X[train_idx], y[train_idx])
    predictions_rf[test_idx] = model.predict(X[test_idx])
    session = groups[test_idx[0]]
    rf_folds.append({
        'Unseen test session': session,
        'Train WAVs': len(train_idx),
        'Test WAVs': len(test_idx),
        'Accuracy': accuracy_score(y[test_idx], predictions_rf[test_idx]),
        'Macro precision': precision_score(y[test_idx], predictions_rf[test_idx],
                                           average='macro', labels=CLASSES, zero_division=0),
        'Macro recall': recall_score(y[test_idx], predictions_rf[test_idx],
                                     average='macro', labels=CLASSES, zero_division=0),
        'Macro F1': f1_score(y[test_idx], predictions_rf[test_idx],
                             average='macro', labels=CLASSES, zero_division=0)
    })
rf_fold_results = pd.DataFrame(rf_folds)

def summarize_model(name, p):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y, p),
        'Macro precision': precision_score(y, p, average='macro', labels=CLASSES, zero_division=0),
        'Macro recall': recall_score(y, p, average='macro', labels=CLASSES, zero_division=0),
        'Macro F1': f1_score(y, p, average='macro', labels=CLASSES, zero_division=0)
    }

comparison = pd.DataFrame([
    summarize_model('SVM (RBF)', predictions_svm),
    summarize_model('Random Forest', predictions_rf),
])
print('=== SVM vs RANDOM FOREST (SAME SESSION SPLITS) ===')
display(comparison.style.format({
    'Accuracy': '{:.2%}', 'Macro precision': '{:.2%}',
    'Macro recall': '{:.2%}', 'Macro F1': '{:.2%}'
}))
comparison.to_csv('/content/svm_vs_random_forest_comparison.csv', index=False)
svm_fold_results.assign(Model='SVM (RBF)').to_csv('/content/svm_fold_scores.csv', index=False)
rf_fold_results.to_csv('/content/random_forest_fold_scores_same_split.csv', index=False)

cm_rf = confusion_matrix(y, predictions_rf, labels=CLASSES)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d', xticks_rotation=30
)
ax.set_title('Random Forest: Same held-out sessions')
plt.tight_layout()
fig.savefig('/content/random_forest_same_split_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

paired_predictions = svm_prediction_table[['wav', 'session', 'actual']].copy()
paired_predictions['SVM predicted'] = predictions_svm
paired_predictions['RF predicted'] = predictions_rf
paired_predictions['SVM correct'] = (y == predictions_svm)
paired_predictions['RF correct'] = (y == predictions_rf)
paired_predictions.to_csv('/content/svm_vs_rf_per_recording.csv', index=False)
print('Recordings where the models disagree:')
display(paired_predictions.loc[paired_predictions['SVM predicted'] != paired_predictions['RF predicted']])
print('These are preliminary results from two sessions and one participant, not generalization estimates to new people.')

## Cell 7 — Save the trained **SVM** and download results

Run ▶. This fits a final SVM on **all 50 WAVs** for later prediction (it is **not** the model used to calculate the accuracy above). It saves the required feature order and all preprocessing inside the `.joblib` model bundle. A ZIP file will download containing the model and both-model comparison reports, metrics, confusion matrices and per-WAV predictions.

In [ ]:
import joblib
from google.colab import files

# Final deployment model, trained on all 50 samples only AFTER held-out evaluation.
final_svm = clone(svm_pipeline)
final_svm.fit(X, y)

svm_bundle = {
    'model': final_svm,
    'model_type': 'SVC_RBF_with_StandardScaler',
    'feature_columns': X_df.columns.tolist(),
    'classes': CLASSES,
    'sample_rate_hz': 48000,
    'carrier_frequencies_hz': CARRIERS,
    'channels_zero_based': CHANNELS,
    'fft_size': FFT_SIZE,
    'hop_size': HOP_SIZE,
    'sideband_low_hz': SIDE_BAND_LOW_HZ,
    'sideband_high_hz': SIDE_BAND_HIGH_HZ,
    'warning': 'Only 50 WAVs from one participant and two closely spaced sessions; needs independent testing.'
}
model_file = Path('/content/svm_gesture_model.joblib')
joblib.dump(svm_bundle, model_file)

output_files = [
    model_file,
    Path('/content/svm_confusion_matrix.png'),
    Path('/content/svm_classification_report.csv'),
    Path('/content/svm_predictions.csv'),
    Path('/content/svm_fold_scores.csv'),
    Path('/content/svm_vs_random_forest_comparison.csv'),
    Path('/content/random_forest_same_split_confusion_matrix.png'),
    Path('/content/random_forest_fold_scores_same_split.csv'),
    Path('/content/svm_vs_rf_per_recording.csv'),
]
zip_filename = Path('/content/ultrasonic_svm_vs_random_forest_results.zip')
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filepath in output_files:
        if not filepath.exists():
            raise FileNotFoundError(f'Missing result file {filepath.name}; run Cells 1–6 first.')
        archive.write(filepath, arcname=filepath.name)

print('SVM model trained and results ZIP ready:', zip_filename.name)
print('Contains the SVM model, CSV metric reports, predictions, and confusion matrices.')
files.download(str(zip_filename))

## Understanding and reporting the results

- **Accuracy:** fraction of 50 recordings correctly classified across the two held-out folds.
- **Macro precision:** average precision across LEFT, RIGHT, TOWARD, AWAY, IDLE (each class weighted equally).
- **Macro F1:** averages the per-class balance of precision and recall; particularly useful if some gestures are harder.
- **Confusion matrix:** rows are actual gestures; columns are predicted gestures. Check which gestures the SVM confuses with each other (especially LEFT).
- **Comparison:** Cell 6 reruns Random Forest with the **same split and features**, so the comparison is meaningful. Differences might be large because the dataset is very small.
- **Important limitation:** both sessions are from one participant close in time; performance on other people, positions, speeds and days is **not established**. You should collect new independent sessions after choosing your model. **Do not tune C or gamma based on these test-session scores**, or you will bias the reported results.

SVM with scaling: [scikit-learn SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).